# Tutorial 7: What can ROC curves conceal when the positive class is rare?

## Briefing for the responsible study group

| Planning item | Allocation |
|---|---|
| Presentation date | Monday 30 November 2026 |
| Prerequisite lecture | Model Evaluation, 20 November 2026 |
| Preparation time after prerequisite | 5 full weekdays, excluding lecture and tutorial dates |
| Format | 15-minute presentation + approximately 15-minute discussion |

**Lecture connection:** Model Evaluation slides 16–23: confusion-matrix rates, ROC curves and AUC; extend these to precision–recall analysis.

This is your group's working notebook. It replaces the requirement to augment a section of an existing course notebook. The supplied cells establish a starting point; your group develops and teaches the investigation.

### Your teaching challenge

Introduce the precision–recall (PR) curve and explain when it offers a more useful view than ROC for rare-positive detection. Your argument must be precise: ROC is not mathematically invalid for imbalanced data, and PR is not universally superior. They expose different aspects of performance.

Use the supplied class-conditional score samples to represent populations with different positive prevalence. Hold the score distribution within each class fixed. The scores are ranking scores between 0 and 1, **not guaranteed calibrated probabilities**. Positive means the rare event of interest, such as fraud or default.

### Core investigation

1. Introduce recall, precision and false-positive rate using the confusion matrix. Explain how moving a score threshold traces ROC and PR curves.
2. Compare prevalence values of 50%, 10% and 1%, keeping class-conditional score distributions fixed. Plot ROC curves and PR curves, and report ROC-AUC and average precision (AP).
3. Explain the prevalence-level no-skill baseline for PR and the 0.5 no-skill ROC-AUC benchmark. Clarify that AP is a particular summary of the PR curve, not simply every possible definition of area under it.
4. At one common score threshold, report expected counts for a hypothetical population of 10,000. Show how many false alarms can coexist with a small false-positive rate.
5. Discuss why PR comparisons across datasets with different prevalence require care. Define which class is positive and explain why that choice matters.

### What classmates should gain

They should understand why a seemingly favourable ROC operating point can still produce mostly false alerts. Verify the relationship precision = prevalence * TPR / (prevalence * TPR + (1-prevalence) * FPR) at one operating point. The formula is undefined if its denominator is zero.

### Boundaries and support

The fixed score pools and weighting helper are supplied. Reweighting avoids introducing sampling noise when prevalence changes: your empirical class-conditional distributions remain exactly the same, so the weighted ROC curves should agree up to numerical precision. These are hypothetical population comparisons; no classifier is retrained. Do not add oversampling, SMOTE, class weighting during training or a separate algorithm benchmark. **Optional:** compare weighted results with random finite samples and explain the extra variability.

## How to turn your investigation into a tutorial

Your group has **15 minutes to present**, followed by approximately **15 minutes of discussion** with the instructor and class. Your audience has attended the prerequisite lecture but has not done your investigation. Help them understand one important question through a focused, reproducible demonstration.

**Required contribution.** Complete the core investigation below, explain why the results occur, and discuss at least one limitation or contrasting condition. Select two or three main figures or tables that make your argument clear. A useful tutorial connects the question, experimental design, evidence and conclusion; a sequence of unexplained code cells is insufficient. You are not expected to build a production system, implement algorithms from scratch, collect external data or run an extensive model search.

**What to submit.** Submit your completed `.ipynb` notebook with explanations and reproducible code, plus any slides or other presentation material you use. Follow the instructor's announced submission deadline. Keep the supplied briefing, but edit, reorganize or replace the working sections as needed. Identify substantial changes to supplied helpers. Record package versions, seeds and relevant settings; restart the kernel and run all cells before submission. Clearly distinguish supplied material from your group's contribution.

**Presentation format is your choice.** You may present from the notebook, create slides, render to HTML, develop an interactive demonstration or combine these. Make the selected format work within 15 minutes and have a fallback for live demonstrations. Creativity earns credit when it improves understanding, engagement or the quality of the investigation. Technical spectacle should be avoided. A clear static plot can be excellent.

**AI assistance is allowed.** You may use AI to help develop code, explanations and visualizations. Your group remains responsible for correctness and must be able to explain the submitted work. Include a short account of the tools used, their main contributions, and how you checked those contributions. Verify at least one important result independently—for example, with a hand calculation, an analytic special case or a comparison with a trusted library function. A long prompt transcript is not required. Using AI is optional and is not itself rewarded.

**Assessment.** This component contributes **15% of the course grade**. Within the tutorial, the criteria are **conceptual understanding (40%)**, **added value for classmates (40%)**, and **quality of communication (20%)**. Added value comes from the investigation and how it develops understanding beyond the lecture and the supplied starter cells. Optional extensions are not required for an excellent grade. All groups should prioritize a sound core investigation over breadth.

**Suggested presentation rhythm:** about 3 minutes for the question and essential concepts, 8 minutes for the demonstration and evidence, and 4 minutes for interpretation and limitations. Prepare one question that invites the class to reason about your findings. Anticipate questions about experimental controls, assumptions and alternative explanations.

## Preliminaries: environment

The course environment provides all libraries that you need to execute the startet codes (see [ReadMe for intallation instructions](https://github.com/stefanlessmann/ESMT_IML)). You can install additional libraries if needed. 


Run the supplied cells first. They provide a starting point, **not a completed tutorial**. Code cells marked `YOUR GROUP'S WORK` are intentionally unfinished and can be replaced or expanded.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from IPython.display import display

SEED = 888
plt.rcParams.update({'figure.figsize': (8, 4.5), 'axes.grid': True,
                     'grid.alpha': 0.2, 'font.size': 11})
print(f'Python {sys.version.split()[0]} | NumPy {np.__version__} | scikit-learn {sklearn.__version__}')

## Preliminaries: supplied data and helpers

Each pool has 2,000 scores. Use `prevalence_weights` as `sample_weight` in scikit-learn curve and metric functions. The weights assign total mass equal to the intended prevalence to positives and the remaining mass to negatives. Weighted confusion proportions can be multiplied by a population size to get expected counts; these need not be integers.

In [ ]:
from scipy.special import expit
from sklearn.metrics import (roc_curve, precision_recall_curve,
                             roc_auc_score, average_precision_score)

rng = np.random.default_rng(SEED)
negative_scores = expit(rng.normal(-1.0, 1.1, 2000))
positive_scores = expit(rng.normal(1.0, 1.1, 2000))
y = np.r_[np.zeros(len(negative_scores), dtype=int),
          np.ones(len(positive_scores), dtype=int)]
scores = np.r_[negative_scores, positive_scores]
prevalences = [0.50, 0.10, 0.01]

def prevalence_weights(y_values, prevalence):
    labels = np.asarray(y_values)
    if not 0 < prevalence < 1 or not np.isin(labels, [0, 1]).all():
        raise ValueError('Use binary labels and a prevalence strictly between 0 and 1')
    n_pos, n_neg = (labels == 1).sum(), (labels == 0).sum()
    if min(n_pos, n_neg) == 0:
        raise ValueError('Both classes must be present')
    return np.where(labels == 1, prevalence / n_pos, (1 - prevalence) / n_neg)

display(pd.DataFrame({'y': y, 'score': scores}).head())
print('Fixed score pools:', len(negative_scores), 'negatives and', len(positive_scores), 'positives')
# Each metric/curve call should use the same weights for its chosen prevalence.

## Investigation 1: Introduce PR through a threshold

Choose one threshold and calculate TPR, FPR and precision using weighted counts. Verify precision with the prevalence formula. Explain the denominator of every rate.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 2: Construct the population comparisons

Create weights for each prevalence and plot both curve families. Compute weighted ROC-AUC and AP, add the appropriate baselines, and explain why the ROC comparison is controlled.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Investigation 3: Translate rates into workload

Convert one operating point into expected counts per 10,000 cases. Explain when precision/recall serves the application and when ROC remains useful. Discuss prevalence and positive-class conventions.

Add your explanation, code and evidence below.

In [ ]:
# YOUR GROUP'S WORK
# Add your investigation here. Split this cell into several cells if helpful.

*Your interpretation of the results: replace this prompt with an explanation.*

## Your teaching conclusions

Replace this text with:

1. A direct answer to the tutorial's central question, supported by your results.
2. Two or three takeaways that classmates should remember.
3. One limitation, counterexample or condition under which the result changes.
4. One discussion question, plus notes on how your group would answer it.

## Contribution and verification record

- **Supplied material retained or changed:** …
- **Your group's experimental and explanatory contributions:** …
- **Independent check of an important result:** …
- **AI tools and their contributions, if any:** …
- **How generated code/explanations were checked:** …
- **Package versions, random seeds and reproduction instructions:** …
- **Group members and contributions:** …

## Before presenting

- [ ] Restarted the kernel and ran all cells successfully.
- [ ] Checked that splits, metrics, units and class labels are correct.
- [ ] Distinguished evidence from interpretation; avoided claims stronger than the experiment supports.
- [ ] Labelled plots and made figures legible for classmates.
- [ ] Rehearsed within 15 minutes and prepared for discussion.
- [ ] Included the completed notebook and any additional presentation material.